# 男女播音员 · 中英混读 tiny-voca 训练
在同一个 Drive 根目录完成 Qwen3-TTS 全量生成后运行本笔记本。默认男女各 31,799 段，共 63,598 段，沿用原句的训练、验证和测试划分。Misaki 提供音素和重音，MMS_FA 为每段录音单独估计音素时长。训练检查点可切换男女声。


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, zipfile, json
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/tiny-voca-mix-qwen3-tts')

# 上传默认输入包后，两个笔记本均可独立恢复全量文本和音素。
import shutil
import zipfile
PACKAGE = ROOT.parent / 'tiny-voca-mix-colab-31799.zip'
if PACKAGE.is_file():
    (ROOT / 'inputs').mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(PACKAGE) as bundle:
        for name in ('corpus.jsonl', 'misaki.jsonl'):
            member = 'tiny-voca-mix-qwen3-tts/inputs/' + name
            dest = ROOT / 'inputs' / name
            temp = dest.with_suffix('.jsonl.part')
            with bundle.open(member) as src, temp.open('wb') as out:
                shutil.copyfileobj(src, out)
            temp.replace(dest)
WORK = Path('/content/tiny-voca-announcers')
DATA = WORK / 'paired'
OUTPUT = ROOT / 'tiny-voca-training'
WORK.mkdir(exist_ok=True)
OUTPUT.mkdir(exist_ok=True)
for item in ['inputs/corpus.jsonl', 'inputs/misaki.jsonl', 'announcers.json']:
    if not (ROOT / item).is_file():
        raise FileNotFoundError(f'缺少 {item}。请解压输入包，并完成 Qwen3-TTS 两种音色的全量生成。')
SMOKE = False
STEPS = 80 if SMOKE else 104000
BATCH_FRAMES = 4000
MAX_BATCH = 16
subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'], check=True)


## 安装独立训练环境
使用 Python 3.11、Torch/TorchAudio 2.8、Misaki 中文 v1.1 和美式英语前端。

In [ ]:
def run(*args, **kwargs):
    subprocess.run(list(map(str,args)),check=True,**kwargs)
run(sys.executable,'-m','pip','install','-q','uv')
PY = WORK/'env/bin/python'
if not PY.exists():
    run('uv','venv','--python','3.11',WORK/'env')
run('uv','pip','install','--python',PY,'--index-url','https://download.pytorch.org/whl/cu128','torch==2.8.0','torchaudio==2.8.0')
requirements = WORK/'requirements-misaki.txt'
requirements.write_text('# Shared Chinese and American English frontend for training and inference.\nmisaki[zh]==0.9.4\npypinyin==0.55.0\npypinyin-dict==0.9.0\njieba==0.42.1\ncn2an==0.5.24\nnum2words==0.5.14\nspacy==3.7.5\nnumpy==1.26.4\ntyper==0.12.5\nclick==8.1.8\nsetuptools<81\nphonemizer-fork==3.3.2\nespeakng-loader==0.2.4\nen-core-web-sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.7.1/en_core_web_sm-3.7.1-py3-none-any.whl\nregex==2026.9.29\n', encoding='utf-8')
run('uv','pip','install','--python',PY,'-r',requirements,'librosa==0.11.0','soundfile==0.13.1')
run(PY,'-c','import torch,torchaudio; assert torch.cuda.is_available(); print(torch.__version__,torch.cuda.get_device_name())')

## 训练代码与双语前端
训练标注和新检查点的推理调用同一份 Misaki 适配代码。

In [ ]:
import base64, io
SRC = WORK/'source'
SRC.mkdir(exist_ok=True)
bundle = '...'
with zipfile.ZipFile(io.BytesIO(base64.b64decode(bundle))) as z:
    z.extractall(SRC)
os.environ['PYTHONPATH']=str(SRC)

In [ ]:
run(PY, '-c', 'from tiny_voca.text import frontend_identity, frontend_fingerprint; print(frontend_identity()["phonemizer"], frontend_fingerprint())')

In [ ]:
(SRC/'preprocess.py').write_text(r'''
"""Align each recording using MMS words/syllables and cache small chunks on Drive."""
import collections, hashlib, json, os, shutil, sys, tarfile
from pathlib import Path
import torch, torchaudio
from torchaudio.pipelines import MMS_FA
from tiny_voca.config import Config
from tiny_voca.prepare import _prepare_row
from tiny_voca.text import units, phonemize, frontend_fingerprint

data, local, drive = map(Path,sys.argv[1:4])
c=Config.from_dict(json.loads((data/"prepare-config.json").read_text()))
local.mkdir(parents=True,exist_ok=True)
key=hashlib.sha256()
for name in ("train","validation","test"):
    key.update((data/(name+".jsonl")).read_bytes())
key.update((frontend_fingerprint()+c.fingerprint).encode())
key.update(Path(__file__).read_bytes())
tag=key.hexdigest()
cache=drive/"preparation"/tag
cache.mkdir(parents=True,exist_ok=True)
model=None

@torch.inference_mode()
def durations(text,wave,frames):
    global model
    if model is None:
        model=MMS_FA.get_model().eval().cuda()
    groups=units(text)
    words=[w for w,_ in groups if w]
    x=torchaudio.functional.resample(wave,24000,16000).unsqueeze(0).cuda()
    emission,_=model(x)
    labels=MMS_FA.get_tokenizer()(words)
    flat=[v for group in labels for v in group]
    if not all(labels) or len(flat)+sum(a==b for a,b in zip(flat,flat[1:]))>emission.size(1):
        raise ValueError("CTC target exceeds emission length")
    spans=iter(MMS_FA.get_aligner()(emission[0],labels))
    scale=frames/emission.size(1)
    centers=[]
    voiced=[]
    for word,phones in groups:
        if word:
            span=next(spans)
            lo,hi=span[0].start*scale,span[-1].end*scale
            for j,(symbol,_) in enumerate(phones):
                centers.append(lo+(j+.5)*(hi-lo)/len(phones))
                voiced.append("/" in symbol)
        else:
            centers.append(None)
            voiced.append(False)
    # Anchor punctuation/boundaries in available gaps. Each pronounced phone gets >= 1 frame.
    if sum(voiced)>frames: raise ValueError("Too many phones")
    centers[0],centers[-1]=0.,float(frames)
    import numpy as np
    indices=[i for i,v in enumerate(centers) if v is not None]
    curve=np.interp(np.arange(len(centers)),indices,[centers[i] for i in indices])
    cuts=np.rint((curve[:-1]+curve[1:])/2).astype(int)
    result=[]
    pos=0
    remaining=sum(voiced)
    for i,cut in enumerate([*cuts,frames]):
        remaining-=voiced[i]
        end=min(frames-remaining,max(pos+int(voiced[i]),int(cut)))
        result.append(end-pos)
        pos=end
    return result

def dump(file,rows):
    file.write_text("".join(json.dumps(r,ensure_ascii=False)+"\n" for r in rows))

torch.set_num_threads(1)
for split in ("train","validation","test"):
    source=data/(split+".jsonl")
    rows=list(map(json.loads,source.read_text().splitlines()))
    (local/split).mkdir(exist_ok=True)
    accepted=[]; rejected=[]
    for offset in range(0,len(rows),128):
        label=f"{split}-{offset:05d}"
        archive=cache/(label+".tar")
        good=local/(label+".jsonl"); bad=local/(label+".rejected.jsonl")
        if archive.exists():
            with tarfile.open(archive) as t: t.extractall(local,filter="data")
        else:
            batch=[]; failures=[]
            for i,row in enumerate(rows[offset:offset+128],offset):
                try:
                    ph=phonemize(row["text"])
                    if (row["frontend"] != frontend_fingerprint() or list(ph.ids) != row["tokens"]
                        or list(ph.stress) != row["stress"]):
                        raise RuntimeError(f"Stored phonemes differ from the runtime frontend for {row['id']}")
                    if len(ph)>c.max_tokens: raise ValueError("Too many tokens")
                    record=_prepare_row(source,0,i,row,c,"text","voice_id",split,local/split)
                    wave=torch.load(local/record["path"],weights_only=True)["wave"]
                    d=durations(row["text"],wave,record["frames"])
                    assert len(d)==len(ph) and sum(d)==record["frames"]
                    record.update(tokens=ph.ids,stress=ph.stress,durations=d,language=row["language"],
                                  frontend=frontend_fingerprint(),annotation=tag)
                    batch.append(record)
                except ValueError as exc:
                    failures.append(dict(id=row["id"],language=row["language"],reason=str(exc)))
            dump(good,batch); dump(bad,failures)
            tmp=archive.with_suffix(".partial")
            with tarfile.open(tmp,"w") as t:
                for path in [good,bad,*[local/r["path"] for r in batch]]:
                    t.add(path,arcname=path.relative_to(local))
            os.replace(tmp,archive)
        accepted.extend(map(json.loads,good.read_text().splitlines()))
        rejected.extend(map(json.loads,bad.read_text().splitlines()))
        print(split,offset+min(128,len(rows)-offset),"/",len(rows),"accepted",len(accepted),flush=True)
    dump(local/(split+".jsonl"),accepted)
    dump(cache/(split+".rejected.jsonl"),rejected)
    for lang in ("zh","en","mixed"):
        n=sum(r["language"]==lang for r in rows)
        used=sum(r["language"]==lang for r in accepted)
        print(split,lang,used,"/",n)
        if not n or used<.8*n: raise ValueError("Inspect rejection log before training")
(local/"annotations.json").write_text(json.dumps(dict(complete=True,fingerprint=tag,acoustic_root=str(local))))
'''.lstrip(), encoding='utf-8')


## 配对男女声录音与全量音素标注
按原句 ID 核对两套生成结果，每个音色使用独立的录音 ID。

In [ ]:
(SRC/'prepare_announcers.py').write_text(r'''
"""Join the two full Qwen runs with the frozen Misaki inputs for Colab training."""
import hashlib
import json
from collections import Counter
from pathlib import Path

SPEAKERS = ('male', 'female')
SPLITS = ('train', 'validation', 'test')
CORPUS_SHA256 = '582ad8a8b4a395bc63e4ad3714ef278ab760e1a126e290478f317cc2a1b5961a'


def load_by_id(path):
    records = {}
    with path.open(encoding='utf-8') as stream:
        for line in stream:
            row = json.loads(line)
            if row['id'] in records:
                raise ValueError(f"Duplicate ID {row['id']} in {path}")
            records[row['id']] = row
    return records


def prepare(root, data, frontend):
    corpus_file = root / 'inputs/corpus.jsonl'
    if hashlib.sha256(corpus_file.read_bytes()).hexdigest() != CORPUS_SHA256:
        raise ValueError('Expected the 31,799-sentence corpus supplied with the notebooks')
    corpus = load_by_id(corpus_file)
    labels = load_by_id(root / 'inputs/misaki.jsonl')
    if labels.keys() != corpus.keys():
        raise ValueError('Misaki annotation IDs differ from the corpus')
    for uid, row in corpus.items():
        label = labels[uid]
        if (any(label[k] != row[k] for k in ('language', 'split', 'text_id'))
                or label['frontend'] != frontend):
            raise ValueError(f'Annotation metadata or frontend differs for {uid}')

    runs = json.loads((root / 'announcers.json').read_text())
    if set(runs) != set(SPEAKERS) or len(set(runs.values())) != len(SPEAKERS):
        raise ValueError('Expected two distinct completed Qwen runs')
    rows = {split: [] for split in SPLITS}
    voices = set()
    references = set()
    for speaker in SPEAKERS:
        run_id = runs[speaker]
        if len(run_id) != 16 or any(c not in '0123456789abcdef' for c in run_id):
            raise ValueError(f'Invalid run ID for {speaker}')
        folder = root / 'runs' / run_id
        recipe_bytes = (folder / 'recipe.json').read_bytes()
        recipe = json.loads(recipe_bytes)
        if (hashlib.sha256(recipe_bytes).hexdigest()[:16] != run_id
                or recipe['corpus_sha256'] != CORPUS_SHA256 or recipe['voice'] in voices
                or recipe['reference_sha256'] in references):
            raise ValueError(f'Incorrect corpus, run or duplicate reference voice for {speaker}')
        voices.add(recipe['voice'])
        references.add(recipe['reference_sha256'])
        recordings = load_by_id(folder / 'corpus.jsonl')
        if recordings.keys() != corpus.keys():
            raise ValueError(f'{speaker} requires one recording of every sentence')
        for uid, sentence in corpus.items():
            record = recordings[uid]
            bucket = hashlib.sha256(uid.encode()).hexdigest()[:2]
            relative = Path('audio', sentence['language'], bucket, uid + '.wav')
            path = folder / relative
            if (any(record[k] != sentence[k] for k in ('text', 'text_id', 'language', 'split'))
                    or record['audio'] != relative.as_posix() or record['voice_id'] != recipe['voice']
                    or record['sample_rate'] != 24000 or record['seconds'] <= 0):
                raise ValueError(f'Recording metadata differs for {speaker}/{uid}')
            if not path.is_file():
                raise FileNotFoundError(path)
            label = labels[uid]
            rows[sentence['split']].append({
                **record, 'id': f'{speaker}-{uid}', 'sentence_id': uid, 'voice_id': speaker,
                'source_id': run_id, 'audio': str(path.resolve()), 'origin': 'qwen3-tts',
                'tokens': label['tokens'], 'stress': label['stress'], 'frontend': frontend,
            })
    data.mkdir(parents=True, exist_ok=True)
    for split, items in rows.items():
        with (data / f'{split}.jsonl').open('w', encoding='utf-8') as stream:
            for row in items:
                print(json.dumps(row, ensure_ascii=False), file=stream)
        print(split, dict(Counter(r['voice_id'] for r in items)), flush=True)
    (data / 'prepare-config.json').write_text(json.dumps({'speakers': SPEAKERS}))
'''.lstrip(), encoding='utf-8')
run(PY, '-c', 'from pathlib import Path; import sys; from prepare_announcers import prepare; from tiny_voca.text import frontend_fingerprint; prepare(Path(sys.argv[1]), Path(sys.argv[2]), frontend_fingerprint())', ROOT, DATA)


## 提取特征与强制对齐
每块完成即存到 Drive。重连时会恢复到本地磁盘。按语种输出接受量；被拒样本的 ID 和原因保存在 preparation 下。

In [ ]:
if SMOKE:
    subset=WORK/'announcers-smoke'
    subset.mkdir(exist_ok=True)
    shutil.copy2(DATA/'prepare-config.json',subset/'prepare-config.json')
    for split in ('train','validation','test'):
        rows=list(map(json.loads,(DATA/(split+'.jsonl')).read_text().splitlines()))
        count=8 if split=='train' else 2
        picked=[r for voice in ('male','female') for lang in ('zh','en','mixed')
                for r in [v for v in rows if v['language']==lang and v['voice_id']==voice][:count]]
        (subset/(split+'.jsonl')).write_text(''.join(json.dumps(r,ensure_ascii=False)+'\n' for r in picked))
    INPUT_DATA=subset
else:
    INPUT_DATA=DATA
FEATURES=WORK/('features-smoke' if SMOKE else 'features')
run(PY,SRC/'preprocess.py',INPUT_DATA,FEATURES,OUTPUT)

## 训练与续训
默认每 500 步保存到 Drive。改变 batch 或其他训练配方时应使用新实验名。中途重连后重跑前面的单元，再运行本单元即可续训。

In [ ]:
import shutil
annotation=json.loads((FEATURES/'annotations.json').read_text())['fingerprint']
experiment=OUTPUT/('model-'+annotation[:12])
experiment.mkdir(exist_ok=True)
shutil.copytree(SRC,experiment/'source',dirs_exist_ok=True,ignore=shutil.ignore_patterns('__pycache__'))
RUN=experiment/'checkpoints'
last=RUN/'last.pt'
args=[PY,'-m','tiny_voca.train','--config',DATA/'prepare-config.json','--device','cuda',
      '--train',FEATURES/'train.jsonl','--validation',FEATURES/'validation.jsonl',
      '--output',RUN,'--steps',STEPS,'--batch-frames',BATCH_FRAMES,'--max-batch',MAX_BATCH,
      '--save-every',500,'--keep-every',10000,'--max-seconds',86400,'--no-preload']
if last.exists():
    args.extend(['--resume',last])
def saved_step():
    if not last.exists(): return 0
    return int(subprocess.check_output([str(PY),'-c','import torch,sys; print(torch.load(sys.argv[1],map_location="cpu",weights_only=True)["step"])',str(last)]))
if saved_step()<STEPS:
    if not last.exists() and RUN.exists() and all(f.name in ('recipe.json','config.json','metrics.jsonl') for f in RUN.iterdir()):
        from datetime import datetime
        RUN.rename(experiment/('interrupted-before-first-save-'+datetime.now().strftime('%Y%m%d%H%M%S')))
    run(*args)
print('已保存步数',saved_step(),RUN)

## 用同一个检查点分别试听男女声


In [ ]:
from IPython.display import Audio, display
annotation=json.loads((FEATURES/'annotations.json').read_text())['fingerprint']
checkpoint=OUTPUT/('model-'+annotation[:12])/'checkpoints/inference.pt'
for voice in ('male', 'female'):
    for lang,text in [('zh','今天最低气温是负十度。'),('en','Could you open the window, please?'),('mixed','这个 weekend 我想去 hiking，你来吗？')]:
        wav=WORK/(voice+'-'+lang+'.wav')
        run(PY,'-m','tiny_voca.infer','--checkpoint',checkpoint,'--voice',voice,'--text',text,'--output',wav)
        print(voice, text)
        display(Audio(filename=str(wav)))